In [ ]:
import pandas as pd
import random
from datetime import timedelta, date
from faker import Faker

# ==============================================================================
# НАСТРОЙКИ И КОНСТАНТЫ
# ==============================================================================

# Инициализация генератора данных
fake = Faker('ru_RU')

# Настройки распределения статусов (используется для генерации ФАКТА)
STATUS_DISTRIBUTION = {
    'В срок': 0.70,      # 70%
    'Досрочно': 0.10,    # 10%
    'Просрочка': 0.20    # 20%
}

# ==============================================================================
# ОБЩИЕ ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ==============================================================================

def generate_random_date(start_date, end_date):
    """Генерирует случайную дату между start_date и end_date."""
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    if days_between_dates <= 0:
        return start_date
    random_number_of_days = random.randrange(days_between_dates + 1)
    return start_date + timedelta(days=random_number_of_days)

def generate_contract_number():
    """Генерирует номер договора в формате №XXX/ГГГГ."""
    num = random.randint(1, 999)
    year = random.randint(2023, 2024)
    return f"№{num:03d}/{year}"

def generate_inn():
    """Генерирует ИНН юридического лица (10 цифр)."""
    return str(random.randint(1000000000, 9999999999))

def get_contract_subject():
    """Возвращает случайный предмет договора."""
    subjects = [
        "Поставка товаров",
        "Оказание услуг",
        "Лицензионный договор",
        "Аренда помещений",
        "Подрядные работы",
        "Консалтинговые услуги",
        "IT-услуги и поддержка",
        "Маркетинговые услуги",
        "Транспортные услуги",
        "Энергоснабжение"
    ]
    return random.choice(subjects)

def get_payment_frequency():
    """Возвращает периодичность платежей."""
    frequencies = ["Разовый", "Ежемесячный", "Ежеквартальный"]
    weights = [0.3, 0.5, 0.2]
    return random.choices(frequencies, weights=weights)[0]

def get_payment_scenario():
    """
    Возвращает сценарий оплаты относительно даты акта.
    """
    scenarios = [
        "Предоплата",        # Платеж ДО акта
        "Оплата в день акта", # Платеж = Дата акта
        "Постоплата"         # Платеж ПОСЛЕ акта (отсрочка)
    ]
    weights = [0.3, 0.2, 0.5] # Постоплата наиболее частая
    return random.choices(scenarios, weights=weights)[0]

# ==============================================================================
# ФУНКЦИИ ДЛЯ ГЕНЕРАЦИИ ФАКТА (СПИСАНИЕ/ПОСТУПЛЕНИЕ С ОТКЛОНЕНИЯМИ)
# ==============================================================================

def calculate_penalty(days_overdue, payment_amount):
    """
    Рассчитывает штраф за просрочку платежа.
    Ставка: 0.1% от суммы платежа за каждый день просрочки.
    """
    if days_overdue <= 0:
        return 0.0
    
    penalty_rate = 0.001  # 0.1% в день
    penalty = payment_amount * penalty_rate * days_overdue
    
    # Минимальный штраф 1000 руб, максимальный 10% от суммы
    penalty = max(1000, min(penalty, payment_amount * 0.1))
    
    return round(penalty, 2)

def assign_payment_status(total_rows):
    """
    Распределяет статусы платежей согласно заданным пропорциям.
    Возвращает список статусов.
    """
    statuses = []
    
    # Рассчитываем количество для каждого статуса
    on_time_count = int(total_rows * STATUS_DISTRIBUTION['В срок'])
    early_count = int(total_rows * STATUS_DISTRIBUTION['Досрочно'])
    overdue_count = total_rows - on_time_count - early_count  # Остаток - просрочка
    
    # Создаем список статусов
    statuses = ['В срок'] * on_time_count + ['Досрочно'] * early_count + ['Просрочка'] * overdue_count
    
    # Перемешиваем для случайного распределения по строкам
    random.shuffle(statuses)
    
    return statuses

def get_deviation_by_status(status):
    """
    Возвращает отклонение в днях в зависимости от статуса.
    """
    if status == 'Досрочно':
        # Досрочно: от -1 до -10 дней
        return random.randint(-10, -1)
    elif status == 'В срок':
        # В срок: от -2 до +2 дней (небольшое отклонение считаем в срок)
        return random.randint(-2, 2)
    else:  # Просрочка
        # Просрочка: от +1 до +30 дней
        return random.randint(1, 30)

def generate_fact_dataset(num_contracts=20, max_rows=200):
    print(f"Генерация факта поступлений ({num_contracts} договоров, макс {max_rows} строк)...")
    print(f"Распределение статусов: {STATUS_DISTRIBUTION}")
    
    # Диапазон года для фактических данных
    fact_start = date(2024, 1, 1)
    fact_end = date(2024, 12, 31)
    
    contracts = []
    all_payments = []
    
    # 1. Сначала генерируем контракты
    for i in range(num_contracts):
        contract_date = generate_random_date(date(2023, 10, 1), date(2024, 1, 15))
        contract_start = contract_date + timedelta(days=random.randint(1, 15))
        
        # Дата окончания
        if random.random() < 0.2:  # 20% бессрочных
            contract_end = date(9999, 12, 31)
            contract_status = "Бессрочный"
        elif random.random() < 0.4:  # 40% продлеваемых
            contract_end = contract_start + timedelta(days=random.randint(365, 730))
            contract_status = "Продлеваемый"
        else:
            contract_end = contract_start + timedelta(days=random.randint(90, 365))
            contract_status = "Срочный"
        
        contract_amount = round(random.uniform(100000, 10000000), 2)
        frequency = get_payment_frequency()
        scenario = get_payment_scenario()
        
        # Расчет суммы платежа в зависимости от периодичности
        if frequency == "Разовый":
            payment_amount = contract_amount
        elif frequency == "Ежемесячный":
            payment_amount = contract_amount / 12
        else:  # Ежеквартальный
            payment_amount = contract_amount / 4
        
        contracts.append({
            'number': generate_contract_number(),
            'date': contract_date,
            'subject': get_contract_subject(),
            'start': contract_start,
            'end': contract_end,
            'status': contract_status,
            'amount': contract_amount,
            'frequency': frequency,
            'scenario': scenario,
            'payment_amount': round(payment_amount, 2),
            'counterparty': fake.company(),
            'inn': generate_inn()
        })
    
    # 2. Собираем все плановые платежи (база для факта)
    planned_payments = []
    
    for contract in contracts:
        if len(planned_payments) >= max_rows:
            break
            
        # Определяем количество платежей в зависимости от периодичности
        if contract['frequency'] == "Разовый":
            num_payments = 1
        elif contract['frequency'] == "Ежемесячный":
            num_payments = min(12, (max_rows - len(planned_payments)))
        else:  # Ежеквартальный
            num_payments = min(4, (max_rows - len(planned_payments)))
        
        for p in range(num_payments):
            if len(planned_payments) >= max_rows:
                break
            
            # Рассчитываем ПЛАНОВУЮ дату платежа
            if contract['frequency'] == "Разовый":
                planned_payment_date = contract['start'] + timedelta(days=random.randint(1, 30))
            elif contract['frequency'] == "Ежемесячный":
                planned_payment_date = contract['start'] + timedelta(days=30 * p)
            else:  # Ежеквартальный
                planned_payment_date = contract['start'] + timedelta(days=90 * p)
            
            # Не выходим за пределы года и срока договора для плановой даты
            if planned_payment_date > fact_end:
                break
            if contract['end'] != date(9999, 12, 31) and planned_payment_date > contract['end']:
                break
            
            # Применяем сценарий оплаты для определения плановой даты акта
            if contract['scenario'] == "Предоплата":
                planned_act_date = planned_payment_date + timedelta(days=random.randint(5, 30))
            elif contract['scenario'] == "Оплата в день акта":
                planned_act_date = planned_payment_date
            else:  # Постоплата
                planned_act_date = planned_payment_date - timedelta(days=random.randint(7, 45))
                if planned_act_date < contract['start']:
                    planned_act_date = contract['start']
            
            planned_payments.append({
                'contract': contract,
                'planned_payment_date': planned_payment_date,
                'planned_act_date': planned_act_date
            })
    
    # 3. Присваиваем статусы согласно распределению
    actual_rows = len(planned_payments)
    statuses = assign_payment_status(actual_rows)
    
    print(f"\n📊 Всего платежей: {actual_rows}")
    print(f"   В срок: {statuses.count('В срок')} ({statuses.count('В срок')/actual_rows*100:.1f}%)")
    print(f"   Досрочно: {statuses.count('Досрочно')} ({statuses.count('Досрочно')/actual_rows*100:.1f}%)")
    print(f"   Просрочка: {statuses.count('Просрочка')} ({statuses.count('Просрочка')/actual_rows*100:.1f}%)")
    
    # 4. Генерируем фактические даты на основе статусов
    for i, payment_info in enumerate(planned_payments):
        contract = payment_info['contract']
        planned_payment_date = payment_info['planned_payment_date']
        planned_act_date = payment_info['planned_act_date']
        
        # Получаем статус для этой строки
        status = statuses[i]
        
        # Получаем отклонение на основе статуса
        deviation_days = get_deviation_by_status(status)
        
        # Рассчитываем фактическую дату платежа
        actual_payment_date = planned_payment_date + timedelta(days=deviation_days)
        
        # Корректируем дату акта (может тоже сдвинуться)
        act_deviation = random.randint(-5, 10)
        actual_act_date = planned_act_date + timedelta(days=act_deviation)
        
        # Не выходим за пределы года
        if actual_payment_date < fact_start:
            actual_payment_date = fact_start
        if actual_payment_date > fact_end:
            actual_payment_date = fact_end
        if actual_act_date < fact_start:
            actual_act_date = fact_start
        if actual_act_date > fact_end:
            actual_act_date = fact_end
        
        # === РАСЧЕТ ПРОСРОЧКИ И ШТРАФОВ ===
        days_overdue = (actual_payment_date - planned_payment_date).days
        penalty_amount = calculate_penalty(days_overdue, contract['payment_amount'])
        
        # НДС 20%
        nds_amount = round(contract['payment_amount'] * 0.2, 2)
        amount_with_nds = round(contract['payment_amount'] + nds_amount, 2)
        
        all_payments.append({
            'Сумма платежа': contract['payment_amount'],
            'Сумма платежа + НДС': amount_with_nds,
            'НДС (20%)': nds_amount,
            'Плановая дата платежа': planned_payment_date,
            'Фактическая дата платежа': actual_payment_date,
            'Отклонение (дней)': days_overdue,
            'Статус оплаты': status,
            'Штраф за просрочку': penalty_amount,
            'Плановая дата акта': planned_act_date,
            'Фактическая дата акта': actual_act_date,
            'Номер договора': contract['number'],
            'Дата договора': contract['date'],
            'Суть договора': contract['subject'],
            'Дата начала действия': contract['start'],
            'Дата окончания действия': contract['end'],
            'Статус договора': contract['status'],
            'Периодичность платежей': contract['frequency'],
            'Сценарий оплаты': contract['scenario'],
            'Сумма договора +НДС': contract['amount'],
            'Контрагент': contract['counterparty'],
            'ИНН контрагента': contract['inn']
        })
    
    # Создаем DataFrame
    df = pd.DataFrame(all_payments)
    
    # Сортируем по фактической дате поступления платежа
    df = df.sort_values(by='Фактическая дата платежа').reset_index(drop=True)
    
    return df, contracts

def save_fact_to_excel(df, filename='cash_inflow_fact_2024.xlsx'):
    try:
        # Сохраняем в Excel
        df.to_excel(filename, index=False)
        print(f"\n✅ Успешно! Файл сохранен: {filename}")
        print(f"📊 Количество строк: {len(df)}")
        print(f"📑 Столбцы: {', '.join(df.columns)}")
        
        # Статистика по данным
        print("\n📈 Статистика факта поступлений:")
        print(f"   Уникальных договоров: {df['Номер договора'].nunique()}")
        print(f"   Общая сумма поступлений: {df['Сумма платежа + НДС'].sum():,.2f} RUB")
        print(f"   Средняя сумма платежа: {df['Сумма платежа + НДС'].mean():,.2f} RUB")
        print(f"   Общая сумма штрафов: {df['Штраф за просрочку'].sum():,.2f} RUB")
        
        print("\n   По статусу оплаты (фактическое распределение):")
        for status in ['Досрочно', 'В срок', 'Просрочка']:
            count = (df['Статус оплаты'] == status).sum()
            pct = count / len(df) * 100
            target_pct = STATUS_DISTRIBUTION.get(status, 0) * 100
            diff = pct - target_pct
            sign = "+" if diff >= 0 else ""
            print(f"      {status}: {count} платежей ({pct:.1f}%, цель: {target_pct:.0f}%, откл: {sign}{diff:.1f}%)")
        
        print("\n   По периодичности:")
        for freq in df['Периодичность платежей'].unique():
            count = (df['Периодичность платежей'] == freq).sum()
            print(f"      {freq}: {count} платежей")
        
        print("\n   По сценарию оплаты:")
        for scenario in df['Сценарий оплаты'].unique():
            count = (df['Сценарий оплаты'] == scenario).sum()
            print(f"      {scenario}: {count} платежей")
        
        # Статистика просрочек
        overdue_df = df[df['Статус оплаты'] == 'Просрочка']
        if len(overdue_df) > 0:
            print(f"\n⚠️  Просроченные платежи: {len(overdue_df)}")
            print(f"   Средняя просрочка: {overdue_df['Отклонение (дней)'].mean():.1f} дней")
            print(f"   Максимальная просрочка: {overdue_df['Отклонение (дней)'].max()} дней")
            print(f"   Минимальная просрочка: {overdue_df['Отклонение (дней)'].min()} дней")
        
        # Статистика досрочных
        early_df = df[df['Статус оплаты'] == 'Досрочно']
        if len(early_df) > 0:
            print(f"\n💚 Досрочные платежи: {len(early_df)}")
            print(f"   Средняя досрочность: {early_df['Отклонение (дней)'].mean():.1f} дней")
        
        # Проверка логики
        print("\n🔒 Проверка логики:")
        errors = 0
        for idx, row in df.iterrows():
            # Штраф должен быть > 0 только при просрочке
            if row['Статус оплаты'] != 'Просрочка' and row['Штраф за просрочку'] > 0:
                errors += 1
            # Штраф должен быть 0 если нет просрочки
            if row['Статус оплаты'] == 'Просрочка' and row['Штраф за просрочку'] == 0:
                errors += 1
            # Отклонение должно соответствовать статусу
            if row['Статус оплаты'] == 'Досрочно' and row['Отклонение (дней)'] >= 0:
                errors += 1
            if row['Статус оплаты'] == 'В срок' and abs(row['Отклонение (дней)']) > 2:
                errors += 1
            if row['Статус оплаты'] == 'Просрочка' and row['Отклонение (дней)'] <= 0:
                errors += 1
        
        if errors == 0:
            print("   ✅ Ошибок логики не найдено.")
        else:
            print(f"   ⚠️ Найдено ошибок логики: {errors}")
        
        # Пример первых строк
        print("\n📋 Пример первых 5 строк:")
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        print(df.head(5).to_string())
        
        # Пример просроченных платежей
        overdue_sample = df[df['Штраф за просрочку'] > 0].head(3)
        if len(overdue_sample) > 0:
            print("\n💰 Пример платежей со штрафами:")
            print(overdue_sample[['Номер договора', 'Контрагент', 'Отклонение (дней)', 'Штраф за просрочку']].to_string())
        
        # Пример досрочных платежей
        early_sample = df[df['Статус оплаты'] == 'Досрочно'].head(3)
        if len(early_sample) > 0:
            print("\n💚 Пример досрочных платежей:")
            print(early_sample[['Номер договора', 'Контрагент', 'Отклонение (дней)']].to_string())
        
    except Exception as e:
        print(f"❌ Ошибка при сохранении файла: {e}")

# ==============================================================================
# ФУНКЦИИ ДЛЯ ГЕНЕРАЦИИ ПЛАНА (БАЗОВЫЕ ДАННЫЕ)
# ==============================================================================

def generate_plan_dataset(num_contracts=20, max_rows=200):
    print(f"Генерация плана поступлений ({num_contracts} договоров, макс {max_rows} строк)...")
    
    # Диапазон года для планирования
    plan_start = date(2024, 1, 1)
    plan_end = date(2024, 12, 31)
    
    contracts = []
    all_payments = []
    
    # 1. Сначала генерируем контракты
    for i in range(num_contracts):
        contract_date = generate_random_date(date(2023, 10, 1), date(2024, 1, 15))
        contract_start = contract_date + timedelta(days=random.randint(1, 15))
        
        # Дата окончания
        if random.random() < 0.2:  # 20% бессрочных
            contract_end = date(9999, 12, 31)
            contract_status = "Бессрочный"
        elif random.random() < 0.4:  # 40% продлеваемых
            contract_end = contract_start + timedelta(days=random.randint(365, 730))
            contract_status = "Продлеваемый"
        else:
            contract_end = contract_start + timedelta(days=random.randint(90, 365))
            contract_status = "Срочный"
        
        contract_amount = round(random.uniform(100000, 10000000), 2)
        frequency = get_payment_frequency()
        scenario = get_payment_scenario()
        
        # Расчет суммы платежа в зависимости от периодичности
        if frequency == "Разовый":
            payment_amount = contract_amount
        elif frequency == "Ежемесячный":
            payment_amount = contract_amount / 12
        else:  # Ежеквартальный
            payment_amount = contract_amount / 4
        
        contracts.append({
            'number': generate_contract_number(),
            'date': contract_date,
            'subject': get_contract_subject(),
            'start': contract_start,
            'end': contract_end,
            'status': contract_status,
            'amount': contract_amount,
            'frequency': frequency,
            'scenario': scenario,
            'payment_amount': round(payment_amount, 2),
            'counterparty': fake.company(),
            'inn': generate_inn()
        })
    
    # 2. Генерируем платежи по контрактам
    rows_generated = 0

    for contract in contracts:
        if rows_generated >= max_rows:
            break
            
        # Определяем количество платежей в зависимости от периодичности
        if contract['frequency'] == "Разовый":
            num_payments = 1
        elif contract['frequency'] == "Ежемесячный":
            num_payments = min(12, (max_rows - rows_generated))
        else:  # Ежеквартальный
            num_payments = min(4, (max_rows - rows_generated))
        
        for p in range(num_payments):
            if rows_generated >= max_rows:
                break
            
            # Рассчитываем дату платежа в пределах года и срока договора
            if contract['frequency'] == "Разовый":
                base_payment_date = contract['start'] + timedelta(days=random.randint(1, 30))
            elif contract['frequency'] == "Ежемесячный":
                base_payment_date = contract['start'] + timedelta(days=30 * p)
            else:  # Ежеквартальный
                base_payment_date = contract['start'] + timedelta(days=90 * p)
            
            # Не выходим за пределы года плана и срока договора
            if base_payment_date > plan_end:
                break
            if contract['end'] != date(9999, 12, 31) and base_payment_date > contract['end']:
                break
            
            # Применяем сценарий оплаты для определения даты акта
            if contract['scenario'] == "Предоплата":
                # Платеж ДО акта (акт через 5-30 дней после платежа)
                payment_date = base_payment_date
                act_date = payment_date + timedelta(days=random.randint(5, 30))
            elif contract['scenario'] == "Оплата в день акта":
                # Платеж = Дата акта
                payment_date = base_payment_date
                act_date = payment_date
            else:  # Постоплата
                # Платеж ПОСЛЕ акта (отсрочка 7-45 дней)
                act_date = base_payment_date
                payment_date = act_date + timedelta(days=random.randint(7, 45))
            
            # Проверка, что даты в пределах года плана
            if payment_date > plan_end:
                payment_date = plan_end
            if act_date > plan_end:
                act_date = plan_end
            
            # НДС 20%
            nds_amount = round(contract['payment_amount'] * 0.2, 2)
            amount_with_nds = round(contract['payment_amount'] + nds_amount, 2)
            
            all_payments.append({
                'Сумма платежа': contract['payment_amount'],
                'Сумма платежа + НДС': amount_with_nds,
                'НДС (20%)': nds_amount,
                'Дата поступления платежа': payment_date,
                'Дата акта': act_date,
                'Номер договора': contract['number'],
                'Дата договора': contract['date'],
                'Суть договора': contract['subject'],
                'Дата начала действия': contract['start'],
                'Дата окончания действия': contract['end'],
                'Статус договора': contract['status'],
                'Периодичность платежей': contract['frequency'],
                'Сценарий оплаты': contract['scenario'],
                'Сумма договора +НДС': contract['amount'],
                'Контрагент': contract['counterparty'],
                'ИНН контрагента': contract['inn']
            })
            
            rows_generated += 1
    
    # Создаем DataFrame
    df = pd.DataFrame(all_payments)
    
    # Сортируем по дате поступления платежа
    df = df.sort_values(by='Дата поступления платежа').reset_index(drop=True)
    
    return df, contracts

def save_plan_to_excel(df, filename='cash_inflow_plan_2024.xlsx'):
    try:
        # Сохраняем в Excel
        df.to_excel(filename, index=False)
        print(f"✅ Успешно! Файл сохранен: {filename}")
        print(f"📊 Количество строк: {len(df)}")
        print(f"📑 Столбцы: {', '.join(df.columns)}")
        
        # Статистика по данным
        print("\n📈 Статистика плана поступлений:")
        print(f"   Уникальных договоров: {df['Номер договора'].nunique()}")
        print(f"   Общая сумма поступлений: {df['Сумма платежа + НДС'].sum():,.2f} RUB")
        print(f"   Средняя сумма платежа: {df['Сумма платежа + НДС'].mean():,.2f} RUB")
        
        print("\n   По периодичности:")
        for freq in df['Периодичность платежей'].unique():
            count = (df['Периодичность платежей'] == freq).sum()
            print(f"      {freq}: {count} платежей")
        
        print("\n   По сценарию оплаты:")
        for scenario in df['Сценарий оплаты'].unique():
            count = (df['Сценарий оплаты'] == scenario).sum()
            print(f"      {scenario}: {count} платежей")
        
        print("\n   По статусу договоров:")
        for status in df['Статус договора'].unique():
            count = (df['Статус договора'] == status).sum()
            print(f"      {status}: {count} записей")
        
        # Проверка логики
        print("\n🔒 Проверка логики:")
        errors = 0
        for idx, row in df.iterrows():
            # Проверка: дата платежа не должна быть раньше начала договора
            if row['Дата поступления платежа'] < row['Дата начала действия']:
                errors += 1
            # Проверка: для предоплаты платеж должен быть раньше акта
            if row['Сценарий оплаты'] == 'Предоплата' and row['Дата поступления платежа'] >= row['Дата акта']:
                errors += 1
            # Проверка: для постоплаты платеж должен быть после акта
            if row['Сценарий оплаты'] == 'Постоплата' and row['Дата поступления платежа'] <= row['Дата акта']:
                errors += 1
        
        if errors == 0:
            print("   ✅ Ошибок логики не найдено.")
        else:
            print(f"   ⚠️ Найдено ошибок логики: {errors}")
        
        # Пример первых строк
        print("\n📋 Пример первых 5 строк:")
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        print(df.head(5).to_string())
        
    except Exception as e:
        print(f"❌ Ошибка при сохранении файла: {e}")

# ==============================================================================
# ОСНОВНАЯ ПРОГРАММА
# ==============================================================================

if __name__ == "__main__":
    print("="*60)
    print("ЗАПУСК ГЕНЕРАЦИИ ПЛАНА ПОСТУПЛЕНИЙ")
    print("="*60)
    # Генерируем данные Плана (20 договоров, макс 200 строк)
    plan_dataframe, plan_contracts = generate_plan_dataset(num_contracts=20, max_rows=200)
    # Сохраняем План в файл
    save_plan_to_excel(plan_dataframe)

    print("\n" + "="*60)
    print("ЗАПУСК ГЕНЕРАЦИИ ФАКТА ПОСТУПЛЕНИЙ")
    print("="*60)
    # Генерируем данные Факта (20 договоров, макс 200 строк)
    fact_dataframe, fact_contracts = generate_fact_dataset(num_contracts=20, max_rows=200)
    # Сохраняем Факт в файл
    save_fact_to_excel(fact_dataframe)
    
    print("\n" + "="*60)
    print("ВСЕ ПРОЦЕССЫ ЗАВЕРШЕНЫ")
    print("="*60)